# Predictive Maintenance using Machine Learning

## Notebook 03 — Feature Engineering & Data Preparation

**Author:** Zun Ding

This notebook prepares the AI4I 2020 Predictive Maintenance dataset for machine learning.

The main objectives are to:

- define the prediction target
- remove identifier and target-leakage variables
- encode categorical features
- engineer meaningful operational features
- separate predictors and target
- create stratified training and test datasets
- prepare reproducible datasets for model training

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

## 1. Load Dataset

The original dataset is loaded again so that the complete data-preparation process remains reproducible and independent of previous notebooks.

In [2]:
df = pd.read_csv("../data/raw/ai4i2020.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 2. Define Target and Remove Leakage

The objective is to predict the overall `Machine failure` variable.

`UDI` and `Product ID` are identifiers rather than direct measurements of machine operating conditions, so they are excluded from the predictors.

The individual failure-mode indicators (`TWF`, `HDF`, `PWF`, `OSF`, and `RNF`) describe specific failure outcomes. Including them when predicting overall machine failure would leak information about the target into the model.

These variables are therefore removed before model training.

In [3]:
target = "Machine failure"

identifier_columns = [
    "UDI",
    "Product ID"
]

leakage_columns = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

drop_columns = identifier_columns + leakage_columns

model_df = df.drop(columns=drop_columns).copy()

model_df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [4]:
print("Original shape:", df.shape)
print("Modelling shape:", model_df.shape)
print("\nColumns used for modelling:")
print(model_df.columns.tolist())

Original shape: (10000, 14)
Modelling shape: (10000, 7)

Columns used for modelling:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']


## 3. Feature Engineering

Several additional features are derived from the original sensor measurements

These features represent relationships between machine operating conditions rather than simply adding arbitrary transformations

The engineered features are:

- **Temperature difference** — difference between process and air temperature
- **Mechanical power** — derived from rotational speed and torque.
- **Torque-speed interaction** — captures the combined operating load represented by torque and rotational speed
- **Tool wear-torque interaction** — combines accumulated tool wear with the mechanical load applied to the machine

These features may help machine-learning models capture operating conditions associated with machine failure

In [5]:
engineered_df = model_df.copy()

# Temperature difference
engineered_df["Temperature difference [K]"] = (
    engineered_df["Process temperature [K]"]
    - engineered_df["Air temperature [K]"]
)

# Convert rotational speed from RPM to angular velocity (rad/s)
angular_velocity = (
    engineered_df["Rotational speed [rpm]"]
    * 2
    * np.pi
    / 60
)

# Mechanical power: Power = Torque × Angular Velocity
engineered_df["Mechanical power [W]"] = (
    engineered_df["Torque [Nm]"] * angular_velocity
)

# Interaction between torque and rotational speed
engineered_df["Torque-speed interaction"] = (
    engineered_df["Torque [Nm]"]
    * engineered_df["Rotational speed [rpm]"]
)

# Interaction between tool wear and torque
engineered_df["Tool wear-torque interaction"] = (
    engineered_df["Tool wear [min]"]
    * engineered_df["Torque [Nm]"]
)

engineered_df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Temperature difference [K],Mechanical power [W],Torque-speed interaction,Tool wear-torque interaction
0,M,298.1,308.6,1551,42.8,0,0,10.5,6951.590560,66382.8,0.0
1,L,298.2,308.7,1408,46.3,3,0,10.5,6826.722724,65190.4,138.9
2,L,298.1,308.5,1498,49.4,5,0,10.4,7749.387543,74001.2,247.0
3,L,298.2,308.6,1433,39.5,7,0,10.4,5927.504659,56603.5,276.5
4,L,298.2,308.7,1408,40.0,9,0,10.5,5897.816608,56320.0,360.0


In [6]:
print("Shape after feature engineering:", engineered_df.shape)

print("\nEngineered features:")
print([
    "Temperature difference [K]",
    "Mechanical power [W]",
    "Torque-speed interaction",
    "Tool wear-torque interaction"
])

Shape after feature engineering: (10000, 11)

Engineered features:
['Temperature difference [K]', 'Mechanical power [W]', 'Torque-speed interaction', 'Tool wear-torque interaction']


## 4. Validate Engineered Features

Before proceeding, the engineered dataset is checked for missing or infinite values introduced during feature construction

In [7]:
engineered_features = [
    "Temperature difference [K]",
    "Mechanical power [W]",
    "Torque-speed interaction",
    "Tool wear-torque interaction"
]

engineered_df[engineered_features].describe()

,Temperature difference [K],Mechanical power [W],Torque-speed interaction,Tool wear-torque interaction
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,10.000630,6279.744953,59967.147040,4314.664550
std,1.001094,1067.418295,10193.093881,2826.567692
min,7.600000,1148.440610,10966.800000,0.000000
25%,9.300000,5561.184484,53105.400000,1963.650000
50%,9.800000,6271.027344,59883.900000,4012.950000
75%,11.000000,7003.002724,66873.750000,6279.000000
max,12.100000,10469.923005,99980.400000,16497.000000


In [8]:
print("Missing values in engineered features:")
print(engineered_df[engineered_features].isnull().sum())

print("\nInfinite values:")
print(
    np.isinf(
        engineered_df[engineered_features].to_numpy()
    ).sum()
)

Missing values in engineered features:
Temperature difference [K]      0
Mechanical power [W]            0
Torque-speed interaction        0
Tool wear-torque interaction    0
dtype: int64

Infinite values:
0


## 5. Separate Predictors and Target

The dataset is separated into:

- `X` — the variables available to the machine-learning models
- `y` — the binary `Machine failure` target

Keeping the target separate prevents it from accidentally being included as a predictor.

In [9]:
X = engineered_df.drop(columns=[target]).copy()
y = engineered_df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPredictor columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

X shape: (10000, 10)
y shape: (10000,)

Predictor columns:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature difference [K]', 'Mechanical power [W]', 'Torque-speed interaction', 'Tool wear-torque interaction']

Target distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64

Target distribution (%):
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


## 6. Stratified Train-Test Split

The data is divided into training and test sets using an 80/20 split.

Because machine failures represent only a small proportion of the observations, stratification is used to preserve approximately the same failure proportion in both datasets.

A fixed random state is used to make the split reproducible.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training predictors:", X_train.shape)
print("Test predictors:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training predictors: (8000, 10)
Test predictors: (2000, 10)
Training target: (8000,)
Test target: (2000,)


In [11]:
print(
    f"Overall failure rate:  {y.mean() * 100:.2f}%"
)

print(
    f"Training failure rate: {y_train.mean() * 100:.2f}%"
)

print(
    f"Test failure rate:     {y_test.mean() * 100:.2f}%"
)

Overall failure rate:  3.39%
Training failure rate: 3.39%
Test failure rate:     3.40%


## 7. Encode Categorical Features

The `Type` variable is categorical and contains three product categories: `L`, `M`, and `H`.

Most machine-learning models require numerical input, so the categorical variable is encoded using one-hot encoding.

One-hot encoding creates separate binary columns for each product type without imposing an artificial numerical order.

In [12]:
categorical_columns = [
    "Type"
]

X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

print("\nEncoded columns:")
print(X_train_encoded.columns.tolist())

Encoded training shape: (8000, 12)
Encoded test shape: (2000, 12)

Encoded columns:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature difference [K]', 'Mechanical power [W]', 'Torque-speed interaction', 'Tool wear-torque interaction', 'Type_H', 'Type_L', 'Type_M']


In [13]:
X_train_encoded.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temperature difference [K],Mechanical power [W],Torque-speed interaction,Tool wear-torque interaction,Type_H,Type_L,Type_M
4058,302.0,310.9,1456,47.2,54,8.9,7196.676675,68723.2,2548.8,0,0,1
1221,297.0,308.3,1399,46.4,132,11.3,6797.736296,64913.6,6124.8,0,0,1
6895,301.0,311.6,1357,45.6,137,10.6,6479.974671,61879.2,6247.2,0,0,1
9863,298.9,309.8,1411,56.3,84,10.9,8318.864043,79439.3,4729.2,0,1,0
8711,297.1,308.5,1733,28.7,50,11.4,5208.456932,49737.1,1435.0,0,1,0


In [14]:
print("Training data types:")
print(X_train_encoded.dtypes)

print("\nMissing values in training data:")
print(X_train_encoded.isnull().sum().sum())

print("\nMissing values in test data:")
print(X_test_encoded.isnull().sum().sum())

Training data types:
Air temperature [K]             float64
Process temperature [K]         float64
Rotational speed [rpm]            int64
Torque [Nm]                     float64
Tool wear [min]                   int64
Temperature difference [K]      float64
Mechanical power [W]            float64
Torque-speed interaction        float64
Tool wear-torque interaction    float64
Type_H                            int64
Type_L                            int64
Type_M                            int64
dtype: object

Missing values in training data:
0

Missing values in test data:
0


## 8. Save Prepared Modelling Data

The processed training and test datasets are saved so that later modelling notebooks can load the exact same data without repeating the entire preparation process.

The target variable is stored together with each dataset for convenience.

In [15]:
train_processed = X_train_encoded.copy()
train_processed[target] = y_train.values

test_processed = X_test_encoded.copy()
test_processed[target] = y_test.values

train_path = "../data/processed/train_processed.csv"
test_path = "../data/processed/test_processed.csv"

train_processed.to_csv(
    train_path,
    index=False
)

test_processed.to_csv(
    test_path,
    index=False
)

print("Saved:")
print(train_path)
print(test_path)

print("\nTraining dataset shape:", train_processed.shape)
print("Test dataset shape:", test_processed.shape)

Saved:
../data/processed/train_processed.csv
../data/processed/test_processed.csv

Training dataset shape: (8000, 13)
Test dataset shape: (2000, 13)


In [16]:
saved_train = pd.read_csv(
    "../data/processed/train_processed.csv"
)

saved_test = pd.read_csv(
    "../data/processed/test_processed.csv"
)

print("Saved training shape:", saved_train.shape)
print("Saved test shape:", saved_test.shape)

saved_train.head()

Saved training shape: (8000, 13)
Saved test shape: (2000, 13)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temperature difference [K],Mechanical power [W],Torque-speed interaction,Tool wear-torque interaction,Type_H,Type_L,Type_M,Machine failure
0,302.0,310.9,1456,47.2,54,8.9,7196.676675,68723.2,2548.8,0,0,1,0
1,297.0,308.3,1399,46.4,132,11.3,6797.736296,64913.6,6124.8,0,0,1,0
2,301.0,311.6,1357,45.6,137,10.6,6479.974671,61879.2,6247.2,0,0,1,0
3,298.9,309.8,1411,56.3,84,10.9,8318.864043,79439.3,4729.2,0,1,0,0
4,297.1,308.5,1733,28.7,50,11.4,5208.456932,49737.1,1435.0,0,1,0,0


In [17]:
final_features = [
    column
    for column in train_processed.columns
    if column != target
]

print(f"Total model features: {len(final_features)}")

for number, feature in enumerate(
    final_features,
    start=1
):
    print(f"{number:>2}. {feature}")

Total model features: 12
 1. Air temperature [K]
 2. Process temperature [K]
 3. Rotational speed [rpm]
 4. Torque [Nm]
 5. Tool wear [min]
 6. Temperature difference [K]
 7. Mechanical power [W]
 8. Torque-speed interaction
 9. Tool wear-torque interaction
10. Type_H
11. Type_L
12. Type_M


In [18]:
target_summary = pd.DataFrame({
    "Training Count": y_train.value_counts().sort_index(),
    "Training Percentage": (
        y_train.value_counts(
            normalize=True
        ).sort_index() * 100
    ),
    "Test Count": y_test.value_counts().sort_index(),
    "Test Percentage": (
        y_test.value_counts(
            normalize=True
        ).sort_index() * 100
    )
})

target_summary.index = [
    "No Failure",
    "Failure"
]

target_summary.round(2)

,Training Count,Training Percentage,Test Count,Test Percentage
No Failure,7729,96.61,1932,96.6
Failure,271,3.39,68,3.4


# Feature Engineering Summary

The AI4I 2020 dataset has now been transformed into a modelling-ready dataset.

## Data Preparation

The following variables were removed:

- `UDI`
- `Product ID`

These variables are identifiers rather than machine operating measurements.

The following failure-mode indicators were also removed:

- `TWF`
- `HDF`
- `PWF`
- `OSF`
- `RNF`

These variables directly describe failure outcomes and could introduce target leakage when predicting overall `Machine failure`.

## Engineered Features

Four additional operational features were created:

1. **Temperature difference [K]**
   - Captures the difference between process and ambient air temperature.

2. **Mechanical power [W]**
   - Combines torque and angular rotational velocity to represent mechanical operating power.

3. **Torque-speed interaction**
   - Captures the interaction between machine torque and rotational speed.

4. **Tool wear-torque interaction**
   - Combines accumulated tool wear with mechanical load.

## Categorical Encoding

The `Type` feature was converted into one-hot encoded variables:

- `Type_H`
- `Type_L`
- `Type_M`

This allows the categorical feature to be used by numerical machine-learning algorithms without assuming an artificial ordering between product categories.

## Train-Test Split

The dataset was divided into:

- 8,000 training observations
- 2,000 test observations

A stratified split was used so that the approximately 3.39% machine-failure rate was preserved in both datasets.

## Final Modelling Data

The processed dataset contains:

- 12 predictor variables
- 1 binary target variable
- no missing values
- no failure-mode leakage variables
- a fixed and reproducible train-test split

The processed datasets are saved in `data/processed/` and will be used consistently throughout the model-development stage.

## Next Step

The next stage will establish baseline classification models and evaluation metrics.

Because machine failure is highly imbalanced, model performance will be assessed using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

Particular attention will be given to **recall for the failure class**, because failing to identify a machine that is actually at risk may be more costly than generating an additional maintenance warning.